In [5]:
import pymupdf4llm

In [1]:
PDF_PATH = r"C:\Users\abubakar\Desktop\agentic_notebook\00_data\parsing_data\A Practitioner's Guide to Language Models.pdf"

In [2]:
def parse_pdf_to_markdown(path: str) -> str:
    return pymupdf4llm.to_markdown(path)

In [6]:
parsed_document =  parse_pdf_to_markdown(PDF_PATH)

rapidocr_api using backend: rapidocr

=== Document parser messages ===
Using RapidOCR for OCR processing.


In [6]:
parsed_document

'# **A Practitioner\'s Guide to Language Models** \n\nAn end-to-end tour from tokens to production serving. \n\n# **Chapter 1: Tokenization** \n\nBefore a language model can process text, that text must be converted into a sequence of integers called tokens. Tokenization is the boundary between human language and the numeric world the model operates in. The choices made here ripple through everything downstream: vocabulary size affects the embedding matrix, token granularity affects sequence length, and segmentation quality affects how well rare words are handled. \n\nEarly systems tokenized on whitespace or characters. Whitespace tokenization produces huge vocabularies and cannot handle words it has never seen. Character tokenization handles any input but produces very long sequences and forces the model to learn spelling from scratch. Modern systems sit between these extremes using subword tokenization, which keeps common words whole while breaking rare words into reusable fragments.

# CHUNKS

# indiviual but this is not correct because in my data many types of data like code,text etc

### RECURSIVE TEXT SPLITTER

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
# Create the splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,      # Maximum characters per chunk
    chunk_overlap=150  # Overlap between chunks
)

chunks = text_splitter.split_text(parsed_document)

chunks[:100]

["# A Practitioner's Guide to Language Models \n\nAn end-to-end tour from tokens to production serving. \n\n## Chapter 1: Tokenization \n\nBefore a language model can process text, that text must be converted into a sequence of integers called tokens. Tokenization is the boundary between human language and the numeric world the model operates in. The choices made here ripple through everything downstream: vocabulary size affects the embedding matrix, token granularity affects sequence length, and segmentation quality affects how well rare words are handled.",
 'Early systems tokenized on whitespace or characters. Whitespace tokenization produces huge vocabularies and cannot handle words it has never seen. Character tokenization handles any input but produces very long sequences and forces the model to learn spelling from scratch. Modern systems sit between these extremes using subword tokenization, which keeps common words whole while breaking rare words into reusable fragments. \n\n##

In [10]:
for i in chunks:
    print(i)
    print("*****"*40)

# **A Practitioner's Guide to Language Models** 

An end-to-end tour from tokens to production serving. 

# **Chapter 1: Tokenization** 

Before a language model can process text, that text must be converted into a sequence of integers called tokens. Tokenization is the boundary between human language and the numeric world the model operates in. The choices made here ripple through everything downstream: vocabulary size affects the embedding matrix, token granularity affects sequence length, and segmentation quality affects how well rare words are handled.
********************************************************************************************************************************************************************************************************
Early systems tokenized on whitespace or characters. Whitespace tokenization produces huge vocabularies and cannot handle words it has never seen. Character tokenization handles any input but produces very long sequences and forces the 

### MARKDOWN TEXT SPLITTERS

In [11]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

# Define headers to split on
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]



markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on
)


docs = markdown_splitter.split_text(parsed_document)

In [12]:
for i, doc in enumerate(docs, 1):
    print(f"\n--- Chunk {i} ---")
    print("Metadata:", doc.metadata)
    print("Content:")
    print(doc.page_content)


--- Chunk 1 ---
Metadata: {'Header 1': "**A Practitioner's Guide to Language Models**"}
Content:
An end-to-end tour from tokens to production serving.

--- Chunk 2 ---
Metadata: {'Header 1': '**Chapter 1: Tokenization**'}
Content:
Before a language model can process text, that text must be converted into a sequence of integers called tokens. Tokenization is the boundary between human language and the numeric world the model operates in. The choices made here ripple through everything downstream: vocabulary size affects the embedding matrix, token granularity affects sequence length, and segmentation quality affects how well rare words are handled.  
Early systems tokenized on whitespace or characters. Whitespace tokenization produces huge vocabularies and cannot handle words it has never seen. Character tokenization handles any input but produces very long sequences and forces the model to learn spelling from scratch. Modern systems sit between these extremes using subword tokenizatio

# Dynamic spliting


asi ma hum apna data ki partition kara gaa ka konsa part text hai aur konsa code hai aur konsa table hai 

### step 1 : identify data type

In [17]:
# CREATING AN SIMPLE DATA CLASS NAMED ELEMENT , THIS WILL BE USED AS A CONTAINER TO STORE PARTITION INFORMATION LIKE WHAT KIND IT IS 
# ACTUAL CONTENT ETC.


from dataclasses import dataclass, field, asdict
from typing import List, Dict, Any

@dataclass
class Element:
    kind: str       # "prose" | "code" | "table"
    text: str
    chapter: str = ""
    section: str = ""

In [18]:
#FUNCTION TO READ DOCUMENT AND PATITION IT ON BASIS OF ITS DATA TYPE.

def partition_markdown(md: str): #FILE LOADED AS MARKDOWN FILE WITH THE HELP OF LOADER
    lines = md.split("\n")  #SPLITTING THE MARKDOWN FILE LINE BY LINE.
    elements: List[Element] = [] #creates an empty list that will store the extracted document parts (e.g., text, code blocks, tables) as Element objects.
    # initialize variables to keep track of the current chapter and section headings while parsing the document.
    chapter = ""
    section = ""

    #STARTING LOOP
    i, n = 0, len(lines)  # initiated two variables i=0 and n= number of lines.
    while i < n:
        stripped = lines[i].strip() # stripping first line

    #CAPTURING HIERARCHY
    #Lets first detect  Markdown headings and updating the current chapter/section context.
        if stripped.startswith("#"): #Checks whether the current line is a Markdown heading (#, ##, ###, etc.).
            level = len(stripped) - len(stripped.lstrip("#")) #Counts how many # symbols are at the beginning to determine the heading level.
            title = stripped.lstrip("#").strip().strip("*").strip()#extracting title.
            if level == 1 or title.lower().startswith(("chapter", "appendix")):
                chapter = title
                section = ""
            else:
                section = title
            i += 1
            continue


        #SEPERATING CODE PART
        if stripped.startswith("```"):
            code_lines = [] #EMPTY LIST TO STORE CODE.
            i += 1 # MOVING TO NEXT LINE AS CODE WILL START FROM NEXT LINE.
            while i < n and not lines[i].strip().startswith("```"): #KEEP READING LINE UNTILL YOU FIND "```" AND STORE THEM IN CODE_LINES
                code_lines.append(lines[i])
                i += 1
            i += 1
            code_text = "\n".join(code_lines)   #COMBINING ALL THE CODE LINES INTO ONE STRING.
            elements.append(Element("code", code_text, chapter, section)) # add new element of type code to the list we created above with type code.
            continue



       #SEPARATING TABLE PART.
        if stripped.startswith("|"):
            table_lines = [] # CREATES AN EMPTY LIST TO STORE ALL THE TABLE ROWS.
            while i < n and lines[i].strip().startswith("|"): #KEEP READING LINES AS LONG AS THEY START WITH '|'
                table_lines.append(lines[i].strip()) #Adds each table row to the list and moves to the next line.
                i += 1
            elements.append(Element("table", "\n".join(table_lines), chapter, section)) # ADD THE NEW ELEMENT OF TYPE TABLE TO THE LIST WE JUST CREATED.
            continue

        #to handle edge cases
        if stripped.startswith(">") and elements and elements[-1].kind == "code":
            cont = stripped.lstrip(">").strip().strip("`")
            elements[-1].text += "\n" + cont
            i += 1
            continue



        #SEPARATING PROSE [TEXTUAL] PART.
        if stripped:
            prose_lines = [] #CREATE AN LIST TO STORE PROSE
            while i < n and lines[i].strip() and not lines[i].strip().startswith(("#", "```", "|")): #CONTINUE COLLECTING LINES TILL YOU REACHED THE
                #END OF THE DOCUMENT, LINE IS NOT EMPTY, LINE IS NOT A HEADING , CODE BLOCK ETC.
                prose_lines.append(lines[i].strip()) #APPEND TO PROSE_LINE
                i += 1
            elements.append(Element("prose", " ".join(prose_lines), chapter, section)) #ADD NEW ELEMENT OF TYPE PROSE.
            continue
 
        i += 1
 
    return elements

In [20]:
partitioned_document=partition_markdown(parsed_document)

In [21]:
merged_chunks = []

for chunk in partitioned_document:
    if (
        merged_chunks
        and chunk.kind in ("prose", "code")
        and merged_chunks[-1].kind == chunk.kind
        and merged_chunks[-1].chapter == chunk.chapter
        and merged_chunks[-1].section == chunk.section
    ):
        # Extend the previous chunk
        merged_chunks[-1].text += "\n" + chunk.text
    else:
        merged_chunks.append(chunk)

# Print merged output
for chunk in merged_chunks:
    print(f"TYPE: {chunk.kind.upper()}")
    print(f"LOCATION: Chapter '{chunk.chapter}' > Section '{chunk.section}'")
    print(f"CONTENT:\n{chunk.text}")
    print("-" * 40)

TYPE: PROSE
LOCATION: Chapter 'A Practitioner's Guide to Language Models' > Section ''
CONTENT:
An end-to-end tour from tokens to production serving.
----------------------------------------
TYPE: PROSE
LOCATION: Chapter 'Chapter 1: Tokenization' > Section ''
CONTENT:
Before a language model can process text, that text must be converted into a sequence of integers called tokens. Tokenization is the boundary between human language and the numeric world the model operates in. The choices made here ripple through everything downstream: vocabulary size affects the embedding matrix, token granularity affects sequence length, and segmentation quality affects how well rare words are handled.
Early systems tokenized on whitespace or characters. Whitespace tokenization produces huge vocabularies and cannot handle words it has never seen. Character tokenization handles any input but produces very long sequences and forces the model to learn spelling from scratch. Modern systems sit between these

## step 2: ROUTING AND SPLITTING.

ab hum jaha pa roting kara gaa ka agr code data ataa hai tu code spliter ka pass ja gaa 

aur agr text a rha hai tu recursive spilter ka pass joo 

aur agr ap table ho tu markdown spliter ka pass joo 



In [22]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, Language

In [12]:
#CREATING SPLITTERS FOR RESPECTIVE TYPES OF DATA
prose_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,      # Set small for testing; increase to ~800 for production
    chunk_overlap=60,
    separators=["\n\n", "\n", ". ", " ", ""]
)


# B. Code Splitter: Uses LangChain's Python-aware rules (looks for def, class, etc.)
code_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON,
    chunk_size= 1500,
    chunk_overlap=0,          
)


def split_code(code_text: str):
    return code_splitter.split_text(code_text)


#KEEPING TABLES AS THEY ARE AS I HAVE SMALL TABLES

In [23]:
# ROUTING AND SPLITTING
def route_and_split(element: Element):
    """Route each element to the appropriate splitter."""

    if element.kind == "prose":
        return prose_splitter.split_text(element.text)

    elif element.kind == "code":
        return split_code(element.text)

    elif element.kind == "table":
        # Keep table as a single chunk
        return [element.text]

    return [element.text]

In [24]:
#CREATING CHUNKS
chunked_documents = []

for element in merged_chunks:
    chunks = route_and_split(element) #creating chunk

    for chunk_text in chunks: # using the chunk to find its meta data from the element and appending it to chunk.
        chunked_documents.append(
            {
                "kind": element.kind,
                "chapter": element.chapter,
                "section": element.section,
                "text": chunk_text,
            }
        )

In [25]:
chunked_documents

[{'kind': 'prose',
  'chapter': "A Practitioner's Guide to Language Models",
  'section': '',
  'text': 'An end-to-end tour from tokens to production serving.'},
 {'kind': 'prose',
  'chapter': 'Chapter 1: Tokenization',
  'section': '',
  'text': 'Before a language model can process text, that text must be converted into a sequence of integers called tokens. Tokenization is the boundary between human language and the numeric world the model operates in. The choices made here ripple through everything downstream: vocabulary size affects the embedding matrix, token granularity affects sequence length, and segmentation quality affects how well rare words are handled.'},
 {'kind': 'prose',
  'chapter': 'Chapter 1: Tokenization',
  'section': '',
  'text': 'Early systems tokenized on whitespace or characters. Whitespace tokenization produces huge vocabularies and cannot handle words it has never seen. Character tokenization handles any input but produces very long sequences and forces the 

In [26]:
# looking at chunked documents.

In [27]:
for chunk in chunked_documents:  # or partitioned_document

    print("\n" + "=" * 100)
    print(f"TYPE    : {chunk['kind'].upper()}")
    print(f"CHAPTER : {chunk['chapter']}")
    print(f"SECTION : {chunk['section']}")

    if chunk["kind"] == "code":
        print("\nCODE:")
        print("-" * 100)
        print(chunk["text"])  # preserves indentation
        print("-" * 100)

    elif chunk["kind"] == "table":
        print("\nTABLE:")
        print("-" * 100)
        print(chunk["text"])
        print("-" * 100)

    else:
        print("\nTEXT:")
        print(chunk["text"])

    print("=" * 100)


TYPE    : PROSE
CHAPTER : A Practitioner's Guide to Language Models
SECTION : 

TEXT:
An end-to-end tour from tokens to production serving.

TYPE    : PROSE
CHAPTER : Chapter 1: Tokenization
SECTION : 

TEXT:
Before a language model can process text, that text must be converted into a sequence of integers called tokens. Tokenization is the boundary between human language and the numeric world the model operates in. The choices made here ripple through everything downstream: vocabulary size affects the embedding matrix, token granularity affects sequence length, and segmentation quality affects how well rare words are handled.

TYPE    : PROSE
CHAPTER : Chapter 1: Tokenization
SECTION : 

TEXT:
Early systems tokenized on whitespace or characters. Whitespace tokenization produces huge vocabularies and cannot handle words it has never seen. Character tokenization handles any input but produces very long sequences and forces the model to learn spelling from scratch. Modern systems sit bet

### CONTEXTUAL RETRIEVAL

In [34]:
import ollama 

CONTEXT_PROMPT = """<document>
{document}
</document>
Here is the chunk we want to situate within the whole document
<chunk>
{chunk}
</chunk>
Please give a short succinct context to situate this chunk within the overall document for the purposes of improving search retrieval of the chunk. 
Answer only with the succinct context and nothing else."""

MODEL= "gemma4:31b-cloud"

#GENERATING SUMMARY.
def generate_context(document: str, chunk: str, model: str = MODEL) -> str:
    prompt = CONTEXT_PROMPT.format(document=document, chunk=chunk)
    resp = ollama.generate(
        model=model,
        prompt=prompt,
        options={"temperature": 0.0},   
    )

    return resp.response.strip()



In [35]:
enriched_documents = []

print(f"Starting enrichment for {len(chunked_documents)} chunks...")

for i, item in enumerate(chunked_documents):
    original_chunk_text = item['text']
    
    print(f"Processing chunk {i + 1}/{len(chunked_documents)}...")
    generated_context = generate_context(document=parsed_document, chunk=original_chunk_text)
    
    # 2. Create a copy of the dictionary so we don't mutate the original data
    enriched_item = item.copy()
    
    # 3. Save the raw context just in case you need it for debugging later
    enriched_item['context'] = generated_context
    
    # 4. Glue the context to the front of the original chunk text
    enriched_item['enriched_text'] = f"{generated_context}\n{original_chunk_text}"
    
    # 5. Append it to our new list
    enriched_documents.append(enriched_item)

print("Enrichment complete!")

# You can now take 'enriched_documents' and embed/store the 'enriched_text' field into your vector database.

Starting enrichment for 201 chunks...
Processing chunk 1/201...
Processing chunk 2/201...
Processing chunk 3/201...
Processing chunk 4/201...
Processing chunk 5/201...
Processing chunk 6/201...
Processing chunk 7/201...
Processing chunk 8/201...
Processing chunk 9/201...
Processing chunk 10/201...
Processing chunk 11/201...
Processing chunk 12/201...
Processing chunk 13/201...
Processing chunk 14/201...
Processing chunk 15/201...
Processing chunk 16/201...
Processing chunk 17/201...
Processing chunk 18/201...
Processing chunk 19/201...
Processing chunk 20/201...
Processing chunk 21/201...
Processing chunk 22/201...
Processing chunk 23/201...
Processing chunk 24/201...
Processing chunk 25/201...
Processing chunk 26/201...
Processing chunk 27/201...
Processing chunk 28/201...
Processing chunk 29/201...
Processing chunk 30/201...
Processing chunk 31/201...
Processing chunk 32/201...
Processing chunk 33/201...
Processing chunk 34/201...
Processing chunk 35/201...
Processing chunk 36/201...

In [36]:
enriched_documents[3]

{'kind': 'prose',
 'chapter': 'Chapter 1: Tokenization',
 'section': '1.1 Byte-Pair Encoding',
 'text': "Byte-Pair Encoding, or BPE, starts from individual characters and repeatedly merges the most frequent adjacent pair into a new symbol. After thousands of merges, common sequences like 'ing' or 'tion' become single tokens, while rare words remain splittable into known pieces. This gives a fixed vocabulary that can still represent any string.\nThe algorithm is simple enough to implement in a few lines. The core loop counts adjacent pairs, finds the most frequent, and records the merge.",
 'context': 'Chapter 1: Tokenization, Section 1.1 Byte-Pair Encoding (BPE). This section explains the mechanism and implementation of subword tokenization in language models.',
 'enriched_text': "Chapter 1: Tokenization, Section 1.1 Byte-Pair Encoding (BPE). This section explains the mechanism and implementation of subword tokenization in language models.\nByte-Pair Encoding, or BPE, starts from indiv